# Classification des signatures thermiques par label (sain ou malade)

In [1]:
import re
import ast
import numpy as np
import pandas as pd

### Importation et traitement des listes

In [2]:
df = pd.read_csv('../data/thermal_signatures.csv')

In [3]:
def clean_and_parse_signature(cell):
    if pd.isna(cell):
        return np.nan
    try:
        # Supprimer les mots-clés comme 'array(', 'dtype=float32', etc.
        cleaned = re.sub(r'array\(', '', cell)
        cleaned = re.sub(r'dtype=\w+', '', cleaned)
        cleaned = re.sub(r'\)', '', cleaned)
        # Remplacer les nans explicites
        cleaned = cleaned.replace('nan', 'None')
        # Tenter d'évaluer
        parsed = ast.literal_eval(cleaned)
        return parsed
    except Exception as e:
        print(f"Erreur parsing : {e}\nCellule brute : {cell}")
        return np.nan

In [4]:
df['Thermal_Signature'] = df['Thermal_Signature'].apply(clean_and_parse_signature)
df['Thermal_Signature'] = df['Thermal_Signature'].apply(lambda x: [np.array(sub) for sub in x] if isinstance(x, list) else x)

In [5]:
# Convertir toutes les chaînes t(a)_C en vraies listes
df['t(a)_C'] = df['t(a)_C'].apply(lambda x: eval(x) if isinstance(x, str) else x)

In [6]:
df

,Patient,File,Side,Label,Tmax_C,t(a)_C,t(e)_C,a_cm,Thermal_Signature
0,66,PAC_16_DN4,left,healthy,33.64,"[32.3, 30.73, 30.32, 30.13, 29.9, 29.7, 29.57,...",26.576218,1,"[[302.9588268, 302.98098159, 303.00335152, 303..."
1,66,PAC_16_DN4,right,healthy,33.12,"[31.3, 30.75, 30.57, 30.46, 30.4, 30.39, 30.16...",26.489057,1,"[[301.62848029, 301.6429305, 301.65753298, 301..."
2,66,PAC_16_DN12,left,healthy,33.53,"[31.62, 30.6, 30.21, 29.9, 29.76, 29.74, 29.75...",26.592819,1,"[[301.81604731, 301.83111251, 301.84633655, 30..."
3,66,PAC_16_DN12,right,healthy,31.81,"[29.57, 28.11, 27.37, 26.44, 26.18, 25.97, 25....",26.557846,1,"[[300.54935196, 300.55577083, 300.56226205, 30..."
4,66,PAC_16_DN7,left,healthy,33.57,"[32.89, 31.25, 30.75, 30.3, 30.16, 30.01, 29.9...",26.602806,1,"[[305.64590747, 305.68036025, 305.71507771, 30..."
...,...,...,...,...,...,...,...,...,...
1390,400,PAC_55_DN4,right,sick,30.87,"[30.05, 29.56, 29.13, 28.85, 28.58, 28.34, 28....",25.000688,1,"[[301.78675347, 301.81014472, 301.8337433, 301..."
1391,400,PAC_55_DN9,right,sick,31.22,"[30.65, 30.34, 29.95, 29.56, 29.33, 29.0, 28.9...",25.098358,1,"[[303.62487673, 303.65585762, 303.68707127, 30..."
1392,400,PAC_55_DN7,right,sick,31.31,"[30.84, 30.47, 30.01, 29.49, 29.26, 28.89, 28....",25.190337,1,"[[304.56311767, 304.59674311, 304.63059694, 30..."
1393,400,PAC_55_DN17,right,sick,31.65,"[31.23, 30.79, 30.58, 30.41, 30.38, 30.12, 29....",25.392738,1,"[[305.54265852, 305.57864002, 305.61484689, 30..."


In [7]:
# Répartition dans Label
print(df['Label'].value_counts())

sick       706
healthy    689
Name: Label, dtype: int64


### Nouveau dataframe avec colonnes sélectionnées

In [8]:
df_filtre = df[['Label', 'Tmax_C', 't(a)_C', 't(e)_C', 'Thermal_Signature']].copy()

In [9]:
# Exploser les listes dans la colonne 'Thermal_Signature' pour avoir une ligne par signature et une valeur pour 't(a)_C'
df_filtre = df_filtre.explode(['t(a)_C', 'Thermal_Signature']).reset_index(drop=True)

In [10]:
df_filtre

,Label,Tmax_C,t(a)_C,t(e)_C,Thermal_Signature
0,healthy,33.64,32.3,26.576218,"[302.9588268, 302.98098159, 303.00335152, 303...."
1,healthy,33.64,30.73,26.576218,"[303.84231819, 303.86918151, 303.89628798, 303..."
2,healthy,33.64,30.32,26.576218,"[306.11147092, 306.14781044, 306.18441781, 306..."
3,healthy,33.64,30.13,26.576218,"[308.41301186, 308.45511248, 308.49745196, 308..."
4,healthy,33.64,29.9,26.576218,"[310.11424081, 310.1581109, 310.20217502, 310...."
...,...,...,...,...,...
13945,sick,31.61,29.96,25.382054,"[314.0888682, 314.1136258, 314.13836262, 314.1..."
13946,sick,31.61,29.84,25.382054,"[314.70500435, 314.72665484, 314.74827625, 314..."
13947,sick,31.61,29.77,25.382054,"[315.24355324, 315.26224106, 315.28089557, 315..."
13948,sick,31.61,29.81,25.382054,"[315.8142144, 315.82951933, 315.84478989, 315...."


## Partie IA

Préparer les données

In [11]:
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

df_filtre = df_filtre.dropna(subset=['Thermal_Signature', 'Tmax_C', 't(a)_C', 't(e)_C', 'Label'])  # Supprimer les lignes incomplètes

# S'assurer que toutes les signatures ont la même taille
signature_lengths = df_filtre['Thermal_Signature'].apply(len)
max_length = signature_lengths.max()
min_length = signature_lengths.min()

# Vérifier que toutes les signatures ont la même longueur
if max_length != min_length:
    print(f"Certaines signatures ont des longueurs différentes ! ({min_length} à {max_length})")
    def fix_length(lst, target_len):
        if len(lst) > target_len:
            return lst[:target_len]
        else:
            return lst + [0.0] * (target_len - len(lst))
    df_filtre['Thermal_Signature'] = df_filtre['Thermal_Signature'].apply(lambda x: fix_length(x, min_length))

# Convertir les colonnes en tableaux et vecteurs colonnes
X_signature = np.array(df_filtre['Thermal_Signature'].tolist())
Tmax_C = df_filtre['Tmax_C'].values.reshape(-1, 1)
T_a_C = df_filtre['t(a)_C'].values.reshape(-1, 1)
T_e_C = df_filtre['t(e)_C'].values.reshape(-1, 1)

# Concaténer les colonnes pour former la matrice X
X = np.hstack((X_signature, Tmax_C, T_a_C, T_e_C))

y = df_filtre['Label']


# Remplacer les NaN dans X par la moyenne de chaque colonne
imputer = SimpleImputer(strategy='mean')
X = imputer.fit_transform(X)

# Encoder les étiquettes de classe
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, random_state=42)

model = DecisionTreeClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

     healthy       0.98      0.98      0.98      2080
        sick       0.98      0.98      0.98      2105

    accuracy                           0.98      4185
   macro avg       0.98      0.98      0.98      4185
weighted avg       0.98      0.98      0.98      4185



Cross validation test

In [13]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

model = DecisionTreeClassifier()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y_encoded, cv=cv, scoring='accuracy')

print(f"\nCross-validated accuracy scores: {scores}")
print(f"Mean accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")



Cross-validated accuracy scores: [0.98064516 0.98709677 0.9827957  0.98315412 0.98351254]
Mean accuracy: 0.9834 (+/- 0.0021)


Cross validation et optimisation des hyperparamètres

In [14]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report

# Split des données avant grid search
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, random_state=42)

# Grille d’hyperparamètres complète
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 5, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# GridSearchCV avec validation croisée
grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Entraînement du modèle avec validation croisée
grid_search.fit(X_train, y_train)

# Résultats de la validation croisée
print("\nBest hyperparameters:")
print(grid_search.best_params_)
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

# Évaluation sur le jeu de test
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\nTest set classification report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Fitting 5 folds for each of 90 candidates, totalling 450 fits

Best hyperparameters:
{'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5}
Best cross-validation accuracy: 0.9839

Test set classification report:
              precision    recall  f1-score   support

     healthy       0.99      0.99      0.99      2080
        sick       0.99      0.99      0.99      2105

    accuracy                           0.99      4185
   macro avg       0.99      0.99      0.99      4185
weighted avg       0.99      0.99      0.99      4185



##### Autres tests

In [15]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier

# Liste des modèles à tester
models = {
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Support Vector Machine": SVC(),
    "Gradient Boosting": GradientBoostingClassifier() # Très long
}

# Boucle d'entraînement et d'évaluation
for name, model in models.items():
    print(f"\nTesting model: {name}")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=le.classes_))



Testing model: Decision Tree
              precision    recall  f1-score   support

     healthy       0.98      0.98      0.98      2080
        sick       0.98      0.98      0.98      2105

    accuracy                           0.98      4185
   macro avg       0.98      0.98      0.98      4185
weighted avg       0.98      0.98      0.98      4185


Testing model: Random Forest
              precision    recall  f1-score   support

     healthy       0.94      0.95      0.94      2080
        sick       0.95      0.94      0.94      2105

    accuracy                           0.94      4185
   macro avg       0.94      0.94      0.94      4185
weighted avg       0.94      0.94      0.94      4185


Testing model: K-Nearest Neighbors
              precision    recall  f1-score   support

     healthy       0.88      0.89      0.88      2080
        sick       0.89      0.88      0.88      2105

    accuracy                           0.88      4185
   macro avg       0.88      0.8

In [16]:
from sklearn.metrics import accuracy_score

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name:25s} Accuracy: {acc:.4f}")

Decision Tree             Accuracy: 0.9799
Random Forest             Accuracy: 0.9438
K-Nearest Neighbors       Accuracy: 0.8815
Support Vector Machine    Accuracy: 0.7508
Gradient Boosting         Accuracy: 0.9637
